# Advanced SQL Examples and Exercises

This notebook contains various examples and exercises focused on more advanced SQL techniques:

1. Modify Data
2. Subqueries
3. Create Views, and Temporary tables (temp tables and CTEs)
4. Combine Query Results with Set Operators
5. Write Queries That Use Window Functions
6. Stored Procedures and Functions
7. Indexes
8. Procedural SQL (Control of Flow Language)

**KEY**

- 🏢 The business scenario
- ✅ Example SQL to address the business requirement
- 💻 A task for you to modify the example SQL
- 📗 Useful reference or guide to review

## About this version: SQLite in your browser

These examples were written in T-SQL, the SQL dialect of Microsoft SQL Server and Azure SQL, for a Sakila database in your own Azure SQL learning environment. This notebook runs them in your browser instead, against a copy of the same Sakila data in **SQLite**. There is nothing to set up, and the answers are in `Advanced_SQL_e-learning_examples_and_exercises_ANSWERS.ipynb` in this lab.

SQLite is a different SQL dialect, so some of the SQL has been translated. Most changes are small:

| T-SQL (Azure SQL) | SQLite (this notebook) |
| --- | --- |
| `SELECT TOP (5) ...` | `SELECT ... LIMIT 5` |
| `first_name + ' ' + last_name` | `first_name \|\| ' ' \|\| last_name` |
| `dbo.category` | `category` (SQLite has no schemas such as `dbo`) |
| `IF OBJECT_ID('t', 'U') IS NOT NULL DROP TABLE t;` | `DROP TABLE IF EXISTS t;` |
| `SELECT ... INTO new_table FROM ...` | `CREATE TABLE new_table AS SELECT ... FROM ...` |
| `#temp_table` | `temp.temp_table`, created with `CREATE TEMP TABLE` |
| `GETDATE()` | `datetime('now')` |
| `DATEADD(day, 1, d)` | `datetime(d, '+1 day')` |
| `FORMAT(EOMONTH(d), 'yyyy-MM')` | `strftime('%Y-%m', d)` |

A few T-SQL features have no direct SQLite equivalent: `MERGE`, stored procedures, user-defined SQL functions, variables and `WHILE` loops, and `IF ... ELSE` / `PRINT`. For those examples the notebook shows the original T-SQL, then does the same job the SQLite way, and says what differs. Look out for the **SQLite note** boxes.

> 📗 SQLite's SQL is documented at https://www.sqlite.org/lang.html

## Setup

Run the cells below first, each time you open the notebook. They install `jupysql`, which lets you run SQL in a notebook cell that starts with `%%sql`, and load a fresh copy of the Sakila database into memory. Because each session starts from a fresh copy, nothing you change here is permanent: run the setup cells again to start over.

In [ ]:
# Hide some harmless but distracting warnings from jupysql
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
import sqlite3
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

# Copy sakila.sqlite into an in-memory database, so every session starts with the original data
source = sqlite3.connect('sakila.sqlite')
con = sqlite3.connect(':memory:', check_same_thread=False)
source.backup(con)
source.close()

# Connect jupysql to that one connection
engine = create_engine('sqlite://', creator=lambda: con, poolclass=StaticPool)
%sql engine

---

### NOTE: You're Not Starting from Scratch!

As you embark on these 18 examples note that you do not have to write SQL code from scratch. Your task is to edit and enhance existing SQL scripts. This is a common and practical skill in data engineering, as modifying and improving existing code is a frequent requirement.

### Running the SQL examples

In the original T-SQL script, `GO` marked the end of each batch of SQL statements, so you could run the script one batch at a time (📗 https://learn.microsoft.com/en-us/sql/t-sql/language-elements/sql-server-utilities-statements-go). In this notebook, each batch is a code cell instead:

1. Run the cells in order, from top to bottom. Some examples build on objects created by the cells before them.
2. To understand each part of the script, run one cell at a time and look at its result before moving on to the next.
3. If an error occurs in one cell, the cells after it can still be run. This is useful for debugging and understanding where your SQL might need adjustments.
4. When a cell holds more than one statement, only the result of the last one is shown.

---

## SECTION 1: Modify Data

In this section, we will explore and practice:

- Learn how to add new records to an existing table using INSERT INTO.
- Learn how to create tables from other tables using SELECT INTO
- Modifying existing records with UPDATE
- Remove records using DELETE FROM
- Use `MERGE` to combine INSERT, UPDATE, and DELETE operations into a single statement.

> 📗 These are some common statements used in Data Manipulation Language: https://learn.microsoft.com/en-us/sql/t-sql/statements/statements?view=sql-server-ver16#data-manipulation-language

> 📗 Guide to five main categories of SQL commands: https://www.geeksforgeeks.org/sql-ddl-dql-dml-dcl-tcl-commands/

---

### 01 INSERT INTO

The INSERT statement in SQL is used to add new rows of data to an existing table.

📗 https://learn.microsoft.com/en-us/sql/t-sql/statements/insert-transact-sql?view=sql-server-ver16#a-inserting-a-single-row-of-data

🏢 **Business Scenario: New movie category**

A Database Administrator (DBA) has been tasked with updating the sakila database for the movie rental company. The company has expanded its catalogue and needs to include new movie categories. The DBA must ensure that the new categories are added correctly without disrupting the existing data.

✅ **Example SQL:** Insert a new category 'Thriller' into the 'category' table.

Count how many rows are currently in the table we will be adding to.

In [ ]:
%%sql
SELECT COUNT(*) as n
FROM category;

Check the structure and typical content of the 'category' table.

In [ ]:
%%sql
SELECT *
FROM category;

Study the 'category' table CREATE TABLE... SQL. In Azure SQL it is at this link: https://github.com/Corndel/sakila/blob/3ae68644a13e714ff33311e122549381a5fdfdea/sql-server-sakila-db/sql-server-sakila-schema.sql#L101

In SQLite, every table's CREATE TABLE... SQL is stored in the `sqlite_master` table, so you can read it with a query:

In [ ]:
%%sql
SELECT sql
FROM sqlite_master
WHERE name = 'category';

From the CREATE TABLE... SQL we can see that 'category_id' is an identity column that "auto-increments" when we insert a new row. Therefore, we don't need to include a value for 'category_id' when we INSERT a new row.

📗 IDENTITY property https://learn.microsoft.com/en-us/sql/t-sql/statements/create-table-transact-sql-identity-property

Also, in the CREATE TABLE... SQL for 'category' there is a CONSTRAINT on the field 'last_update' where its DEFAULT value will be today's date using the getdate() function.

📗 DEFAULT value https://learn.microsoft.com/en-us/sql/relational-databases/tables/specify-default-values-for-columns

📗 getdate() https://learn.microsoft.com/en-us/sql/t-sql/functions/getdate-transact-sql

> **SQLite note:** SQLite writes these differently. `category_id INTEGER PRIMARY KEY AUTOINCREMENT` does the job of `IDENTITY` (📗 https://www.sqlite.org/autoinc.html), and `DEFAULT CURRENT_TIMESTAMP` does the job of `DEFAULT (getdate())`.

We therefore only need to INSERT a value for 'name'. In this case, 'Thriller'.

📗 INSERT https://learn.microsoft.com/en-us/sql/t-sql/statements/insert-transact-sql

In [ ]:
%%sql
INSERT INTO category (name)
VALUES ('Thriller');

Confirm the insertion by counting rows again.

In [ ]:
%%sql
SELECT COUNT(*) as n
FROM category;

View the row for the inserted category.

In [ ]:
%%sql
SELECT *
FROM category
WHERE name = 'Thriller';

Delete the newly added row for cleanup.

In [ ]:
%%sql
DELETE FROM category
WHERE name = 'Thriller';

Confirm the deletion by counting rows again.

In [ ]:
%%sql
SELECT COUNT(*) as n
FROM category;

💻 **Task:** Modify the existing INSERT statement to add two new categories instead of one: Mystery and Romance.

- Ensure that both new categories are added in a single INSERT INTO statement.
- After modifying the INSERT statement, update the subsequent SELECT FROM WHERE statements to verify both insertions.
- Finally, modify the DELETE statement to delete both newly added categories.

📗 How to insert multiple rows of data https://learn.microsoft.com/en-us/sql/t-sql/statements/insert-transact-sql?view=sql-server-ver16#b-inserting-multiple-rows-of-data

---

### 02 INSERT INTO with SELECT

This technique combines data selection and insertion, allowing you to insert data from one or more tables into a new table by selecting, joining and filtering the necessary data with a SELECT statement.

🏢 **Business Scenario: Marketing Team's Request for Actor Promotion Analysis**

The marketing team is planning a promotional campaign. They want to focus on actors who have significant screen presence, defined as appearing in more than 5 films. This example involves creating a 'TopActors' table to aid the marketing team in their analysis and campaign planning. Using the 'actor' and 'film_actor' tables they have asked for a new table with the actor's first name, last name, the actor ID and the total number of films.

✅ **Example SQL:** Use 'actor' and 'film_actor' tables to create a new table to store actors who have appeared in more than 5 films.

Check if the 'TopActors' table already exists and drop it if necessary.

In [ ]:
%%sql
DROP TABLE IF EXISTS TopActors;

Now create the TopActors table.

In [ ]:
%%sql
CREATE TABLE TopActors (
    ActorID INT,
    FirstName VARCHAR(45),
    LastName VARCHAR(45),
    TotalFilms INT
);

Review the structure of the 'actor' and 'film_actor' tables that between them have the information we need.

In [ ]:
%%sql
SELECT *
FROM actor
LIMIT 5;

In [ ]:
%%sql
SELECT *
FROM film_actor
LIMIT 5;

Join and display the data that will be inserted into 'TopActors'. This step allows us to eyeball and check the data is correct before actual insertion.

In [ ]:
%%sql
SELECT
    a.actor_id, -- The unique identifier for each actor
    a.first_name,
    a.last_name,
    COUNT(fa.film_id) AS TotalFilms -- Count of films each actor has appeared in
FROM
    actor a
JOIN
    film_actor fa ON a.actor_id = fa.actor_id -- Joining with 'film_actor' table to count film appearances
GROUP BY
    a.actor_id, a.first_name, a.last_name -- Grouping results by actor ID, first name, and last name
HAVING
    COUNT(fa.film_id) > 5; -- Filtering to include only actors with more than 5 film appearances

After reviewing the data, proceed with the insertion into 'TopActors'.

In [ ]:
%%sql
INSERT INTO TopActors (ActorID, FirstName, LastName, TotalFilms)
SELECT
    a.actor_id,
    a.first_name,
    a.last_name,
    COUNT(fa.film_id) AS TotalFilms
FROM
    actor a
JOIN
    film_actor fa ON a.actor_id = fa.actor_id
GROUP BY
    a.actor_id, a.first_name, a.last_name
HAVING
    COUNT(fa.film_id) > 5;

💻 **Task:** Modify the query to insert actors into 'TopActors' who have appeared in more than 10 films, instead of more than 5. Change the HAVING clause in both the SELECT statement used for review and the INSERT INTO statement to reflect this new criteria.

---

### 03 SELECT INTO

SELECT INTO in SQL creates a new table and populates it with data selected from another table, offering a quick way to copy and transform data.

🏢 **Business Scenario: Creating Temporary Tables for Ad-Hoc Customer Activity Analysis**

The customer service department wants to conduct an ad-hoc analysis of customer activity for July 2005. Rather than adding to the permanent schema of the database, they require a temporary table to assess activity. This approach avoids the overhead of managing schema changes and allows for quick, disposable analysis.

✅ **Example SQL:** create quick table for ad-hoc analysis

Use SELECT INTO to quickly create a table with the necessary data without altering the database schema permanently. The temporary table 'ActiveCustomersJuly2005' will be used for this one-time analysis and can be disposed of afterwards. This method is less complex and avoids the need for multiple statements (CREATE TABLE and INSERT INTO), making it a good choice for temporary or one-off tasks.

> **SQLite note:** SQLite has no `SELECT ... INTO`. The SQLite way to create and fill a table in one statement is `CREATE TABLE ... AS SELECT ...`. The original T-SQL was:
>
> ```sql
> SELECT customer_id, COUNT(rental_id) AS rental_count
> INTO ActiveCustomersJuly2005
> FROM rental
> WHERE rental_date BETWEEN '2005-07-01' AND '2005-07-31'
> GROUP BY customer_id
> HAVING COUNT(rental_id) > 0;
> ```

Ensure the ActiveCustomersJuly2005 table doesn't already exist.

In [ ]:
%%sql
DROP TABLE IF EXISTS ActiveCustomersJuly2005;

Create a table to identify active customers in July 2005.

In [ ]:
%%sql
CREATE TABLE ActiveCustomersJuly2005 AS
SELECT
    customer_id,
    COUNT(rental_id) AS rental_count
FROM
    rental
WHERE
    rental_date BETWEEN '2005-07-01' AND '2005-07-31'
GROUP BY
    customer_id
HAVING
    COUNT(rental_id) > 0;

Check the structure and some data from the newly created ActiveCustomersJuly2005 table.

In [ ]:
%%sql
SELECT *
FROM ActiveCustomersJuly2005
LIMIT 5;

💻 **Task:** Modify the SELECT INTO statement (here, `CREATE TABLE ... AS SELECT`) to include not only the 'customer_id' and 'rental_count', but also the last rental date for each customer in July 2005. This will require adding a MAX function on the 'rental_date' column in the SELECT clause.

📗 Here is a simple example of using the MAX function to help you https://learn.microsoft.com/en-us/sql/t-sql/functions/max-transact-sql?view=sql-server-ver16#a-simple-example

---

### 04 UPDATE

The UPDATE statement in SQL is used to modify existing records in a table, allowing precise changes to specific fields based on defined conditions using WHERE.

🏢 **Business Scenario: Updating Staff Records for Legal Name Change**

The human resources department has been notified that a staff member's legal name has changed. Accurate staff records are crucial for payroll and legal documentation. The staff member, previously recorded under a different name "Jon", has submitted the necessary legal documents to update their name in the company records to "John". You have been told their staff id is 2.

✅ **Example SQL:** Change first name of one staff member

Update the staff member's first name in the 'staff' table to reflect the legal name change. Ensure the change is accurately reflected in the system to maintain up-to-date records.

First, check the current information of the staff member.

In [ ]:
%%sql
SELECT *
FROM staff
WHERE staff_id = 2;

Update the first name of the staff member with staff_id = 2. The staff member has legally changed their name to 'John'.

In [ ]:
%%sql
UPDATE staff
SET first_name = 'John'
WHERE staff_id = 2;

Confirm the update by checking the updated information.

In [ ]:
%%sql
SELECT *
FROM staff
WHERE staff_id = 2;

💻 **Task:** Modify the UPDATE statement to also change the 'store_id' for the staff member with staff_id = 2 to 1, reflecting their move to a different store. This will involve also updating the 'store_id' column in the SET clause to a fixed value of 1.

📗 Here is an example of updating multiple columns to help you: https://learn.microsoft.com/en-us/sql/t-sql/queries/update-transact-sql?view=sql-server-ver16#b-updating-multiple-columns

---

### 05 UPDATE with table joins

The UPDATE statement in SQL can also be used with JOIN clauses to modify data in a table based on values in another table.

🏢 **Business Scenario: Enhancing Rental Strategy for Classic Films**

The marketing department has decided to implement a new strategy to boost the rental rates of classic films. This strategy includes increasing the rental duration for films in the 'Classics' category to provide customers with more value. A longer rental period is expected to make these timeless films more attractive to customers who enjoy the genre.

✅ **Example SQL:** Update the rental duration for all films categorised as 'Classics' in the 'film' table. This change is aimed at increasing the appeal of classic films, hence potentially increasing their rental frequency. The update requires a join with the 'film_category' and 'category' tables to identify the films that fall under the 'Classics' category.

First, review the current rental duration for a few classic films. We need to join the 'film' table with 'film_category' and 'category' tables because the category name ('Classics') we're filtering on is in the 'category' table. The 'film_category' table acts as a bridge between 'film' and 'category' tables.

In [ ]:
%%sql
SELECT
    f.film_id,
    f.title,
    f.rental_duration
FROM film f
INNER JOIN film_category fc ON f.film_id = fc.film_id
INNER JOIN category c ON fc.category_id = c.category_id
WHERE c.name = 'Classics';

Update the rental duration for classic films to 7 days. The update is based on the category, so the joins are necessary to access the category information.

> **SQLite note:** SQLite supports `UPDATE ... FROM`, but the table being updated cannot be listed again in the `FROM` clause. So `film` is joined to the other tables in the `WHERE` clause instead. The original T-SQL was:
>
> ```sql
> UPDATE film
> SET rental_duration = 7
> FROM film f
> INNER JOIN film_category fc ON f.film_id = fc.film_id
> INNER JOIN category c ON fc.category_id = c.category_id
> WHERE c.name = 'Classics';
> ```
>
> 📗 https://www.sqlite.org/lang_update.html#update_from

In [ ]:
%%sql
UPDATE film
SET rental_duration = 7
FROM film_category fc
INNER JOIN category c ON fc.category_id = c.category_id
WHERE film.film_id = fc.film_id
  AND c.name = 'Classics';

Confirm the update by rechecking the rental durations for classic films. Again, joining with 'film_category' and 'category' to filter by category name.

In [ ]:
%%sql
SELECT
    f.film_id,
    f.title,
    f.rental_duration
FROM film f
INNER JOIN film_category fc ON f.film_id = fc.film_id
INNER JOIN category c ON fc.category_id = c.category_id
WHERE c.name = 'Classics';

💻 **Task:** Modify the UPDATE statement to change the rental duration for classic films and set the rental rate to a flat rate of 1.99. This requires updating the 'rental_rate' column in the SET clause to 1.99, along with changing the rental duration.

📗 Here is an example of updating multiple columns to help you: https://learn.microsoft.com/en-us/sql/t-sql/queries/update-transact-sql?view=sql-server-ver16#b-updating-multiple-columns

📗 And here are examples of applying multiplication https://learn.microsoft.com/en-us/sql/t-sql/language-elements/multiply-transact-sql?view=sql-server-ver16#examples

---

### 06 MERGE

MERGE in SQL enables the merging of two tables by applying INSERT, UPDATE, or DELETE operations based on a set of conditions. It is particularly useful for synchronising two tables by applying changes from a source table to a target table.

🏢 **Business Scenario: Updating a staging table**

You have been asked to create and test SQL code to update a staging table of rental records when the rental table changes.

✅ **Example SQL:**

1. Create and populate a staging table with current rentals data
2. Create a temporary copy of the rentals table then simulate changes to that data (i.e. as if time has passed and customer transactions changed rows and values in the database)
3. Use the MERGE statement to synchronise the staging table with the updated rentals table

Note: In real-world scenarios, staging tables are often located in a separate database or even on a different server. This separation is done for reasons such as security, performance optimisation, and to avoid accidental modifications to production data. However, for simplicity in this exercise, we are performing these tasks within the same database.

Step 1: Check and Drop Existing Temporary Table if it exists. Then create a temporary copy of the Rental Table so we can safely simulate changes to the source data.

> **SQLite note:** In T-SQL this step was `SELECT * INTO #temp_rental FROM rental;`, which also copies the IDENTITY property of `rental_id`, so new rows get a `rental_id` automatically. SQLite's `CREATE TABLE ... AS SELECT` does not copy keys, so we declare the temporary table first, with `rental_id` as an auto-incrementing key, and then copy the rows into it.

In [ ]:
%%sql
DROP TABLE IF EXISTS temp.temp_rental;

CREATE TEMP TABLE temp_rental (
    rental_id INTEGER PRIMARY KEY AUTOINCREMENT,
    rental_date DATETIME,
    inventory_id INT,
    customer_id INT,
    return_date DATETIME,
    staff_id INT,
    last_update DATETIME
);

INSERT INTO temp.temp_rental
SELECT * FROM rental;

Step 2: Check and Drop Existing Staging Table if it exists, then Create a New Staging Table.

In [ ]:
%%sql
DROP TABLE IF EXISTS STG_rental;

CREATE TABLE STG_rental (
    rental_id INT,
    rental_date DATETIME,
    inventory_id INT,
    customer_id INT,
    return_date DATETIME,
    staff_id INT,
    last_update DATETIME
);

Step 3: Populate Staging Table with Current Data from Rental for a Baseline Setup.

In [ ]:
%%sql
INSERT INTO STG_rental
SELECT * FROM rental;

Step 4: Simulate data changes on the temporary rentals table.

Insert new fake records into the temporary rental table.

In [ ]:
%%sql
INSERT INTO temp.temp_rental (rental_date, inventory_id, customer_id, return_date, staff_id, last_update)
VALUES
(datetime('now'), 1, 10, NULL, 1, datetime('now')),
(datetime('now'), 2, 25, NULL, 1, datetime('now'));

Update Existing Records in the Temporary Rental Table.

In [ ]:
%%sql
UPDATE temp.temp_rental
SET return_date = datetime(rental_date, '+1 day'),
    last_update = datetime('now')
WHERE rental_id IN (1, 2);

Step 5: Synchronise the data changes just made in the source data to the staging table using MERGE. Update 'STG_rental' with the latest data from the temporary rental table.

> **SQLite note:** SQLite has no `MERGE` statement. The original T-SQL was:
>
> ```sql
> MERGE INTO STG_rental AS target
> USING #temp_rental AS source
> ON target.rental_id = source.rental_id
>
> WHEN MATCHED AND source.last_update > target.last_update THEN
>     UPDATE SET
>         target.rental_date = source.rental_date,
>         target.inventory_id = source.inventory_id,
>         target.customer_id = source.customer_id,
>         target.return_date = source.return_date,
>         target.staff_id = source.staff_id,
>         target.last_update = source.last_update
>
> WHEN NOT MATCHED BY TARGET THEN
>     INSERT (rental_id, rental_date, inventory_id, customer_id, return_date, staff_id, last_update)
>     VALUES (source.rental_id, source.rental_date, source.inventory_id, source.customer_id, source.return_date, source.staff_id, source.last_update)
>
> WHEN NOT MATCHED BY SOURCE THEN
>     DELETE
>
> OUTPUT $action AS Action,
>     INSERTED.rental_id AS NewRentalID,
>     DELETED.rental_id AS OldRentalID,
>     ...;
> ```
>
> In SQLite, each `WHEN` clause becomes its own statement, run in the cells below: an `UPDATE ... FROM` for `WHEN MATCHED`, an `INSERT ... SELECT ... WHERE NOT EXISTS` for `WHEN NOT MATCHED BY TARGET`, and a `DELETE ... WHERE NOT EXISTS` for `WHEN NOT MATCHED BY SOURCE`. Each statement's `RETURNING` clause lists the rows it changed, as `OUTPUT` did. Two differences: `MERGE` does all three in one statement (so it succeeds or fails as a whole), and `RETURNING` shows only the new values of updated rows, where `OUTPUT` showed both the old (`DELETED`) and new (`INSERTED`) values.
>
> 📗 https://www.sqlite.org/lang_returning.html

WHEN MATCHED: update staging rows whose source row has changed since.

In [ ]:
%%sql
UPDATE STG_rental AS target
SET rental_date = source.rental_date,
    inventory_id = source.inventory_id,
    customer_id = source.customer_id,
    return_date = source.return_date,
    staff_id = source.staff_id,
    last_update = source.last_update
FROM temp.temp_rental AS source
WHERE target.rental_id = source.rental_id
  AND source.last_update > target.last_update
RETURNING 'UPDATE' AS Action,
    rental_id AS NewRentalID,
    rental_date AS NewRentalDate,
    inventory_id AS NewInventoryID,
    customer_id AS NewCustomerID,
    return_date AS NewReturnDate,
    staff_id AS NewStaffID,
    last_update AS NewLastUpdate;

WHEN NOT MATCHED BY TARGET: insert source rows that are not yet in the staging table.

In [ ]:
%%sql
INSERT INTO STG_rental (rental_id, rental_date, inventory_id, customer_id, return_date, staff_id, last_update)
SELECT source.rental_id, source.rental_date, source.inventory_id, source.customer_id, source.return_date, source.staff_id, source.last_update
FROM temp.temp_rental AS source
WHERE NOT EXISTS (
    SELECT 1 FROM STG_rental AS target WHERE target.rental_id = source.rental_id
)
RETURNING 'INSERT' AS Action,
    rental_id AS NewRentalID,
    rental_date AS NewRentalDate,
    inventory_id AS NewInventoryID,
    customer_id AS NewCustomerID,
    return_date AS NewReturnDate,
    staff_id AS NewStaffID,
    last_update AS NewLastUpdate;

WHEN NOT MATCHED BY SOURCE: delete staging rows that are no longer in the source.

In [ ]:
%%sql
DELETE FROM STG_rental
WHERE NOT EXISTS (
    SELECT 1 FROM temp.temp_rental AS source WHERE source.rental_id = STG_rental.rental_id
)
RETURNING 'DELETE' AS Action,
    rental_id AS OldRentalID,
    rental_date AS OldRentalDate,
    inventory_id AS OldInventoryID,
    customer_id AS OldCustomerID,
    return_date AS OldReturnDate,
    staff_id AS OldStaffID,
    last_update AS OldLastUpdate;

💻 **Task:**

1. In Step 4 add a 'DELETE FROM... WHERE' on 'temp.temp_rental' to also simulate the deletion of rental records. Delete rental_id's 11 and 12.
2. Then after running the MERGE statements again (from Step 1), inspect the output returned to confirm those two records were deleted.

---

## SECTION 2: 07 Subqueries

Subqueries in SQL allow you to perform nested queries, where you can use the result of an inner query within an outer query.

Alternatives to subqueries include:

1. Temporary Tables: Complex queries can be simplified by storing intermediate results in temporary tables.
2. Common Table Expressions (CTEs): CTEs offer a more readable and potentially more performant alternative to subqueries. We will look at an example of CTEs shortly.
3. Views: Creating a view is another way to simplify more complex queries like this. We will look at carrying out exactly this task with a View in the next example.

Choosing between subqueries and these alternatives depends on factors like the specific use case, performance, data complexity, and code readability.

📗 Subqueries https://learn.microsoft.com/en-us/sql/relational-databases/performance/subqueries

🏢 **Scenario:**

The Movie Selection team wants to analyze the popularity of films within each category to:

1. Tailor marketing strategies and promote popular films
2. Optimize inventory management based on demand

They require a list of all films in each category, ordered by the number of rentals.

✅ **Example SQL:** Using a subquery, develop a SQL query to list rented films in each movie category in order of descending rental volume.

Step 1: Calculate the total number of rentals for each film. This subquery establishes the foundation for identifying the rental count of each film. We join the rental and inventory tables to correlate each film with its rental count.

In [ ]:
%%sql
SELECT
    i.film_id,
    COUNT(r.rental_id) AS rental_count
FROM
    rental r
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
GROUP BY
    i.film_id;

Step 2: List all films in each category, ordered by rental count. In this step, we use the results from Step 1. We join the subquery with film_category, category, and film tables to bring in the category names, film titles, and rental rates. The results are ordered by category name, rental count (descending), and film title, providing a clear overview of film popularity within each category.

In [ ]:
%%sql
SELECT
    c.name AS category_name,
    f.title AS film_title,
    r.rental_count
FROM
    (SELECT
        i.film_id,
        COUNT(r.rental_id) AS rental_count
    FROM
        rental r
        INNER JOIN inventory i ON r.inventory_id = i.inventory_id
    GROUP BY
        i.film_id) r
    INNER JOIN film_category fc ON r.film_id = fc.film_id
    INNER JOIN category c ON fc.category_id = c.category_id
    INNER JOIN film f ON r.film_id = f.film_id
ORDER BY
    c.name,
    r.rental_count DESC,
    f.title;

💻 **Task:** Modify the subquery to only include films with 30 or more rentals.

---

## SECTION 3: Create Views, and Temporary tables (temp tables and CTEs)

### 08 VIEWS

VIEWS in SQL are virtual tables that represent the result of a predefined SQL query.

📗 https://learn.microsoft.com/en-us/sql/relational-databases/views/views

Let's use the same business scenario from the subquery example above and see how it can be performed using a view instead.

✅ **Example SQL:** Using a view, develop a SQL query to list rented films in each movie category in order of descending rental volume.

1. Create a View to calculate the total number of rentals for each film.

In [ ]:
%%sql
DROP VIEW IF EXISTS film_rental_counts;

In [ ]:
%%sql
CREATE VIEW film_rental_counts AS
SELECT
    i.film_id,
    COUNT(r.rental_id) AS rental_count
FROM
    rental r
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
GROUP BY
    i.film_id;

2. Inspect the View to check that the total number of rentals for each film is correctly calculated.

In [ ]:
%%sql
SELECT *
FROM film_rental_counts
ORDER BY film_id
LIMIT 5;

3. Create a query on the view to join the View with the film_category, category, and film tables to identify the most popular film in each category.

In [ ]:
%%sql
SELECT
    c.name AS category_name,
    f.title AS film_title,
    frc.rental_count
FROM
    film_rental_counts frc
    INNER JOIN film_category fc ON frc.film_id = fc.film_id
    INNER JOIN category c ON fc.category_id = c.category_id
    INNER JOIN film f ON frc.film_id = f.film_id
ORDER BY
    c.name,
    frc.rental_count DESC,
    f.title;

---

### 09 TEMPORARY TABLES

Temporary tables can be a useful way to create an intermediate step we don't need to keep so that our logic is easier to read (e.g. compared to nested sub-query).

📗 https://learn.microsoft.com/en-us/sql/t-sql/statements/create-table-transact-sql?view=sql-server-ver16#temporary-tables

Revisiting the same business scenario from the subquery and view methods this time we'll use a temporary table.

✅ **Example SQL:** Using a temp table, develop a SQL query to list rented films in each movie category in order of descending rental volume.

> **SQLite note:** In T-SQL a temporary table's name starts with `#`, as in `#film_rental_counts`, and it was created with `SELECT ... INTO #film_rental_counts`. In SQLite you create one with `CREATE TEMP TABLE`, and it lives in the `temp` schema, so we refer to it as `temp.film_rental_counts`. That also keeps it apart from the view of the same name we created above. Like T-SQL's, SQLite's temporary tables disappear when the connection closes.
>
> 📗 https://www.sqlite.org/lang_createtable.html

1. Create and populate the Temporary Table with the total number of rentals for each film.

In [ ]:
%%sql
DROP TABLE IF EXISTS temp.film_rental_counts;

CREATE TEMP TABLE film_rental_counts AS
SELECT
    i.film_id,
    COUNT(r.rental_id) AS rental_count
FROM
    rental r
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
GROUP BY
    i.film_id;

2. Inspect the Temporary Table to check that the total number of rentals for each film is correctly calculated.

In [ ]:
%%sql
SELECT *
FROM temp.film_rental_counts
LIMIT 5;

3. Create a query on the temporary table to join it with the film_category, category, and film tables to identify the most popular film in each category.

In [ ]:
%%sql
SELECT
    c.name AS category_name,
    f.title AS film_title,
    frc.rental_count
FROM
    temp.film_rental_counts frc
    INNER JOIN film_category fc ON frc.film_id = fc.film_id
    INNER JOIN category c ON fc.category_id = c.category_id
    INNER JOIN film f ON frc.film_id = f.film_id
ORDER BY
    c.name,
    frc.rental_count DESC,
    f.title;

---

### 10 COMMON TABLE EXPRESSIONS (CTEs)

CTEs provide a way to define a temporary result set that you can reference within a SELECT, INSERT, UPDATE, or DELETE statement. They can make complex queries more readable and maintainable, especially when replacing multiple nested subqueries or multiple temporary tables.

📗 https://learn.microsoft.com/en-us/sql/t-sql/queries/with-common-table-expression-transact-sql

Revisiting the same business scenario from the subquery, view, and temporary table examples, we now demonstrate the use of a CTE.

CTEs offer advantages over views and temporary tables in complex querying scenarios. They are temporary, exist only during query execution, and help break down complex queries into simpler parts. CTEs can enhance readability, maintainability, and allow for recursive operations, making them powerful for intricate queries.

✅ **Example SQL:** Using a CTE, develop a SQL query to list rented films in each movie category in order of descending rental volume.

In [ ]:
%%sql
WITH film_rental_counts AS (
    SELECT
        i.film_id,
        COUNT(r.rental_id) AS rental_count
    FROM
        rental r
        INNER JOIN inventory i ON r.inventory_id = i.inventory_id
    GROUP BY
        i.film_id
)
SELECT
    c.name AS category_name,
    f.title AS film_title,
    frc.rental_count
FROM
    film_rental_counts frc
    INNER JOIN film_category fc ON frc.film_id = fc.film_id
    INNER JOIN category c ON fc.category_id = c.category_id
    INNER JOIN film f ON frc.film_id = f.film_id
ORDER BY
    c.name,
    frc.rental_count DESC,
    f.title;

💻 **Task:** Having run the same query using subqueries, temporary tables, views, and CTEs, consider the following:

1. Which method do you find most readable and maintainable?
2. In what scenarios might you prefer one method over the others?
3. How do CTEs compare to the other methods in terms of performance and flexibility?

---

## SECTION 4: Combine Query Results with Set Operators

Demonstrations of using UNION, INTERSECT, and EXCEPT.

### 11 UNION

UNION in SQL is useful for when you need to combine the results of two or more SELECT statements into a single result set, while eliminating duplicate rows. This is particularly handy in scenarios where data needs to be gathered from different tables or datasets with a similar structure but stored separately, ensuring a distinct and unified view of the data.

UNION removes duplicate rows from the result set, while UNION ALL keeps all rows, including duplicates. If the source tables are known to have no duplicates, or if duplicates are acceptable in the result set, UNION ALL can be used for better performance since it doesn't need to remove duplicates.

📗 https://learn.microsoft.com/en-us/sql/t-sql/language-elements/set-operators-union-transact-sql

🏢 **Business Scenario: Create combined list of names from two sources**

For a special event, a list of names is required from both customers and staff, with indicators to distinguish between the two groups.

✅ **Example SQL:** Create a unified list of names from the 'customer' and 'staff' tables, including an indicator column to identify the source table.

> **SQLite note:** T-SQL joins strings with `+`. SQLite uses `||`, so `first_name + ' ' + last_name` becomes `first_name || ' ' || last_name`.

Step 1: Test extracting names from customers with an indicator for the person type.

In [ ]:
%%sql
SELECT
    first_name || ' ' || last_name AS name,
    'Customer' AS person_type
FROM
    customer;

Step 2: Test extracting names from staff with an indicator for the person type.

In [ ]:
%%sql
SELECT
    first_name || ' ' || last_name AS name,
    'Staff' AS person_type
FROM
    staff;

Step 3: Combine Results Using UNION. Append the names from customers and staff into a single list, removing any duplicate names.

In [ ]:
%%sql
SELECT
    first_name || ' ' || last_name AS name,
    'Customer' AS person_type
FROM
    customer
UNION
SELECT
    first_name || ' ' || last_name AS name,
    'Staff' AS person_type
FROM
    staff
ORDER BY
    person_type DESC,
    name;

💻 **Task:**

1. Replace UNION with UNION ALL in the final SELECT statement. Observe any differences in the result set.
2. Consider a scenario where a person is both a customer and a staff member. How would using UNION vs UNION ALL affect the result set in this case?

---

### 12 INTERSECT AND EXCEPT

INTERSECT AND EXCEPT are useful for when you need to perform direct comparisons between two data sets to identify common or exclusive elements. These operators compare distinct rows between the result sets of two queries.

📗 https://learn.microsoft.com/en-us/sql/t-sql/language-elements/set-operators-except-and-intersect-transact-sql

🏢 **Business Scenario:**

As part of a strategic analysis for marketing and content curation, our movie streaming service aims to understand the crossover and uniqueness of actors among different film genres. Specifically, we are interested in exploring the actor overlap between 'Action' and 'Comedy' categories. This insight will guide us in making decisions for targeted marketing campaigns, genre-specific promotions, and understanding actor versatility which is valuable for future film acquisitions and recommendations.

✅ **Example SQL:** Identify actors who are common to both 'Action' and 'Comedy' film categories as well as those who are exclusive to each category. This data will help in understanding actor crossover and exclusivity, enhancing our content and marketing strategies.

While subqueries with IN/NOT IN, complex joins with Group By, and temporary tables or CTEs could be used for this task, they often lead to more complex and less intuitive queries. INTERSECT and EXCEPT in this use case are more concise.

Step 1: Find Common Actors (INTERSECT). Identify actors who have appeared in both 'Action' and 'Comedy' film categories, providing insight into actor versatility and crossover appeal. The JOIN with the film_category and category tables is necessary to determine the category of each film.

In [ ]:
%%sql
SELECT actor_id
FROM film_actor
INNER JOIN film_category ON film_actor.film_id = film_category.film_id
INNER JOIN category ON film_category.category_id = category.category_id
WHERE category.name = 'Action'
INTERSECT
SELECT actor_id
FROM film_actor
INNER JOIN film_category ON film_actor.film_id = film_category.film_id
INNER JOIN category ON film_category.category_id = category.category_id
WHERE category.name = 'Comedy';

Step 2: Find Exclusive Actors for 'Action' Category (EXCEPT). Identify actors unique to the 'Action' category, highlighting exclusivity and potential for targeted genre-specific promotions.

In [ ]:
%%sql
SELECT actor_id
FROM film_actor
INNER JOIN film_category ON film_actor.film_id = film_category.film_id
INNER JOIN category ON film_category.category_id = category.category_id
WHERE category.name = 'Action'
EXCEPT
SELECT actor_id
FROM film_actor
INNER JOIN film_category ON film_actor.film_id = film_category.film_id
INNER JOIN category ON film_category.category_id = category.category_id
WHERE category.name = 'Comedy';

Step 3: Find Exclusive Actors for 'Comedy' Category (EXCEPT). Identify actors unique to the 'Comedy' category, aiding in understanding the genre-specific talent pool for potential content curation.

In [ ]:
%%sql
SELECT actor_id
FROM film_actor
INNER JOIN film_category ON film_actor.film_id = film_category.film_id
INNER JOIN category ON film_category.category_id = category.category_id
WHERE category.name = 'Comedy'
EXCEPT
SELECT actor_id
FROM film_actor
INNER JOIN film_category ON film_actor.film_id = film_category.film_id
INNER JOIN category ON film_category.category_id = category.category_id
WHERE category.name = 'Action';

💻 **Task:** Alter the INTERSECT query to also include the names of the actors, not just their IDs. This will require you to join with the actor table.

---

## SECTION 5: Write Queries That Use Window Functions

📗 https://learn.microsoft.com/en-us/sql/t-sql/queries/select-over-clause-transact-sql

Window functions in SQL are useful when we want to perform calculations across a set of table rows that are somehow related to the current row. This includes tasks like running totals, rankings, or moving averages. Let's now explore examples of how these functions can be applied to real-world data scenarios.

### 13 Ranking Functions

Ranking Functions are useful when we want to assign a rank to each row within a partition of a result set, based on a specified column's values. They assign ranks to rows within a partition based on the order specified in the ORDER BY clause of the OVER clause. They are particularly helpful in scenarios where we need to compare rows to each other, like identifying top-performing categories or items.

📗 https://learn.microsoft.com/en-us/sql/t-sql/functions/ranking-functions-transact-sql

🏢 **Business Scenario:**

In our film rental business, we are interested in understanding the performance of different film categories based on rental revenue. This insight will help us in making informed decisions about inventory management and promotional activities.

✅ **Example SQL:** Create a query that uses a CTE to calculate total rental revenue for each film category and then applies a RANK window function to order categories by revenue.

> **SQLite note:** Azure SQL stores `amount` as an exact `DECIMAL`. SQLite stores it as a floating-point number, so a `SUM` can come out as, say, 4756.9800000000005. `ROUND(..., 2)` is added around sums of money in this notebook to show them to the penny, as Azure SQL did.

Step 1: Create CTE to Calculate Total Revenue per Category. The CTE is used to simplify the query by first calculating the total revenue for each category.

Step 2: Apply RANK Window Function. Rank film categories based on total rental revenue.

In [ ]:
%%sql
-- Step 1: Create CTE to Calculate Total Revenue per Category.
WITH CategoryRevenue AS (
   SELECT
       c.name AS Category,
       ROUND(SUM(p.amount), 2) AS TotalRevenue
   FROM
       rental r
       INNER JOIN inventory i ON r.inventory_id = i.inventory_id
       INNER JOIN film f ON i.film_id = f.film_id
       INNER JOIN film_category fc ON f.film_id = fc.film_id
       INNER JOIN category c ON fc.category_id = c.category_id
       INNER JOIN payment p ON r.rental_id = p.rental_id
   GROUP BY
       c.name
)

-- Step 2: Apply RANK Window Function.
SELECT
   Category,
   TotalRevenue,
   RANK() OVER (ORDER BY TotalRevenue DESC) AS Rank
FROM
   CategoryRevenue
ORDER BY
   Rank;

💻 **Tasks:**

1. Add a new column in the CTE that counts the number of films in each category. Then, include this count in the final SELECT statement.
   📗 COUNT aggregate function: https://learn.microsoft.com/en-us/sql/t-sql/functions/count-transact-sql
2. Modify the ORDER BY clause in the window function to first rank categories by the number of films (added in the previous step), and then by TotalRevenue.
3. Change the RANK() function to DENSE_RANK() and observe how the rankings change if there were ties in the TotalRevenue. While there are no ties, would it make sense to use DENSE_RANK() in case ties occurred in later data updates?
   Explanation of DENSE_RANK(): https://learn.microsoft.com/en-us/sql/t-sql/functions/dense-rank-transact-sql

---

### 14 LAG FUNCTION

The LAG function is useful when you want to easily access data from a previous row in the same result set without the need for self-joins or subqueries. It's particularly effective in time-series analysis, such as comparing monthly sales, tracking changes over periods, or identifying trends in sequential data.

📗 https://learn.microsoft.com/en-us/sql/t-sql/functions/lag-transact-sql

🏢 **Business Scenario:**

Our film rental business requires an analysis of monthly revenue trends. Comparing each month's revenue with the previous month can provide valuable insights.

✅ **Example SQL:** Calculate total payment amount for each month and display alongside the total from the previous month.

Step 1: Aggregate Total Payments by Month. Aggregate payments from the 'payment' table on a monthly basis.

Step 2: Use LAG to Fetch Previous Month's Total. The LAG function is used to get the total amount from the previous month.

> **SQLite note:** The T-SQL version labelled each month with `FORMAT(EOMONTH(payment_date), 'yyyy-MM')`. SQLite has neither function; `strftime('%Y-%m', payment_date)` gives the same year-month label.
>
> 📗 https://www.sqlite.org/lang_datefunc.html

In [ ]:
%%sql
-- Step 1: Aggregate Total Payments by Month.
WITH MonthlyTotals AS (
    SELECT
        strftime('%Y-%m', payment_date) AS Month,
        ROUND(SUM(amount), 2) AS TotalAmount
    FROM
        payment
    GROUP BY
        strftime('%Y-%m', payment_date)
)
-- Step 2: Use LAG to Fetch Previous Month's Total.
SELECT
    Month,
    TotalAmount,
    LAG(TotalAmount, 1, 0) OVER (ORDER BY Month) AS PreviousMonthTotal
FROM
    MonthlyTotals
ORDER BY
    Month;

💻 **Tasks:**

1. Update the ORDER BY clause in the LAG function to explore how different sorting orders (e.g., DESC instead of ASC) impact the PreviousMonthTotal calculation.
2. Extend the CTE to include a count of the total number of transactions for each month, then include this count in your final SELECT statement. This will provide additional context to the revenue figures.

---

## Section 6: Stored Procedures and functions

Stored procedures and functions encapsulate complex business logic, optimise performance, and enhance security. Both lead to code reuse and maintenance efficiency.

Stored procedures are ideal for operations involving multiple steps and can return zero or more results. They are particularly useful for:

- Complex data manipulations
- Repetitive tasks
- Scenarios where security and encapsulation of business logic are important

Functions are designed to compute and return a single value (scalar functions) or a table (table-valued functions). They are often used in SQL statements for:

- Specific value computations
- Encapsulating reusable logic that can be called from multiple queries or stored procedures

The choice between stored procedures and functions depends on the nature of the operation: use stored procedures for complex processes that may involve multiple steps and can return zero or more results. Use functions for specific value computations or for encapsulating reusable logic that returns a single value or a table.

> **SQLite note:** SQLite is a database engine built into the program that uses it, so it has no stored procedures and no `CREATE FUNCTION`. That logic lives in the program instead: here, in Python. The two examples below show the original T-SQL, then the Python equivalent.

---

### 15 STORED PROCEDURES

STORED PROCEDURES are useful for when we want to centralise complex operations, manage permissions efficiently, and improve performance through pre-compiled execution plans. They are ideal for repetitive tasks, complex data manipulations, and scenarios where security and encapsulation of business logic are important.

📗 https://learn.microsoft.com/en-us/sql/relational-databases/stored-procedures/stored-procedures-database-engine

🏢 **Business Scenario:**

Our film rental business needs insights on the most popular films based on rental frequency and total revenue. This information is crucial for inventory management, marketing strategies, and purchasing decisions.

✅ **Example SQL:** Create a stored procedure to generate a report of top-performing films based on total rentals and total revenue.

The original T-SQL created and ran the stored procedure like this:

```sql
-- Step 1: Drop Existing Stored Procedure if It Exists
IF EXISTS (SELECT * FROM sys.objects WHERE type = 'P' AND name = 'sp_TopPerformingFilms')
    DROP PROCEDURE sp_TopPerformingFilms;
GO

-- Step 2: Create the Stored Procedure for the Report
CREATE PROCEDURE sp_TopPerformingFilms
    @TopN INT = 10 -- Parameter to specify the number of top films to return
AS
BEGIN
    SET NOCOUNT ON;

    -- Query to find top N performing films based on rental count and total revenue
    SELECT TOP (@TopN)
        f.film_id,
        f.title,
        COUNT(r.rental_id) AS TotalRentals,
        SUM(p.amount) AS TotalRevenue
    FROM rental r
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
    INNER JOIN film f ON i.film_id = f.film_id
    INNER JOIN payment p ON r.rental_id = p.rental_id
    GROUP BY f.film_id, f.title
    ORDER BY TotalRevenue DESC, TotalRentals DESC;
END;
GO

-- Step 3: Execute the Stored Procedure to Retrieve the Report
EXEC sp_TopPerformingFilms @TopN = 5;
```

> **SQLite note:** Here the procedure becomes a Python function with the same name and parameter. It holds the same query, with `LIMIT :TopN` in place of `TOP (@TopN)`, and runs it with pandas. `:TopN` is a query parameter: SQLite fills in the value you pass, the same job `@TopN` did in T-SQL.

Step 2: Create the "stored procedure" (a Python function) for the report.

In [ ]:
import pandas as pd

def sp_TopPerformingFilms(TopN=10): # Parameter to specify the number of top films to return
    # The logic encapsulated here can be reused and modified independently of the code that calls it.
    # Query to find top N performing films based on rental count and total revenue
    query = """
    SELECT
        f.film_id,
        f.title,
        COUNT(r.rental_id) AS TotalRentals,
        ROUND(SUM(p.amount), 2) AS TotalRevenue
    FROM rental r
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
    INNER JOIN film f ON i.film_id = f.film_id
    INNER JOIN payment p ON r.rental_id = p.rental_id
    GROUP BY f.film_id, f.title
    ORDER BY TotalRevenue DESC, TotalRentals DESC
    LIMIT :TopN;
    """
    return pd.read_sql_query(query, con, params={'TopN': TopN})

Step 3: Execute the "stored procedure" to retrieve the report. By calling the function, we get the benefits of reusable, parameterised logic described above.

In [ ]:
sp_TopPerformingFilms(TopN=5)

💻 **Tasks:**

1. Update the stored procedure (here, the Python function) to return the average rental revenue per film along with the existing columns. This will provide more depth to the analysis.
2. Change the ORDER BY clause to first sort by the new average rental revenue column in descending order, then by TotalRentals.
3. Modify the stored procedure to add a parameter that also allows filtering films by a minimum rental count threshold. This will add flexibility to the report generation.

📗 Stored procedure example with two parameters: https://learn.microsoft.com/en-us/sql/relational-databases/stored-procedures/create-a-stored-procedure?view=sql-server-ver16

---

### 16 Functions

Functions are useful when we want to perform a specific task, such as calculations or processing logic, and return a result. While scalar functions, like the one shown here, return a single value, SQL Server also supports table-valued functions which return a set of rows. Additionally, aggregate functions return a single value calculated from a set of input values. In Azure Data Studio or the Azure portal, the Functions folder of the database holds these different function types.

📗 https://learn.microsoft.com/en-us/sql/t-sql/statements/create-function-transact-sql

🏢 **Business Scenario:**

Our film rental business needs to determine late fees for rentals that exceed the allowed rental duration. The late fee should be calculated based on the number of days the rental exceeds the allowed duration. This assists in billing and reinforces rental policies.

✅ **Example SQL:** Create a function to calculate late fees based on the rental date, return date, and rental duration.

The original T-SQL created the scalar function like this:

```sql
-- Step 1: Check and Drop Existing Function if It Exists
IF object_id('fn_CalculateLateFee') IS NOT NULL
   DROP FUNCTION fn_CalculateLateFee;
GO

-- Step 2: Create the Scalar Function
CREATE FUNCTION dbo.fn_CalculateLateFee
(
    @RentalDate DATETIME,
    @ReturnDate DATETIME,
    @RentalDuration INT,
    @RatePerDay DECIMAL(4,2)
)
RETURNS DECIMAL(10,2)
AS
BEGIN
    DECLARE @DaysOverdue INT;
    SET @DaysOverdue = DATEDIFF(DAY, DATEADD(DAY, @RentalDuration, @RentalDate), @ReturnDate);

    IF @DaysOverdue > 0
        RETURN @DaysOverdue * @RatePerDay;

    RETURN 0;
END;
```

> **SQLite note:** SQLite lets a program add its own SQL functions (📗 https://docs.python.org/3/library/sqlite3.html#sqlite3.Connection.create_function). So Step 2 writes the function in Python, with the same name and parameters, and registers it with the database connection. SQL can then call it like any built-in function. The query in Step 3 is unchanged apart from dropping `dbo.` and using `||` to join strings in place of `CONCAT`.

Step 2: Create the scalar function.

In [ ]:
from datetime import datetime, timedelta

def fn_CalculateLateFee(RentalDate, ReturnDate, RentalDuration, RatePerDay):
    # A rental that has not been returned has no return date (NULL), so no late fee yet.
    # (In T-SQL, DATEDIFF gives NULL, the IF test fails and the function returns 0.)
    if ReturnDate is None:
        return 0
    due_date = datetime.fromisoformat(RentalDate) + timedelta(days=RentalDuration)
    # Like T-SQL's DATEDIFF(DAY, ...), count the calendar days between the two dates
    DaysOverdue = (datetime.fromisoformat(ReturnDate).date() - due_date.date()).days

    if DaysOverdue > 0:
        return round(DaysOverdue * RatePerDay, 2)

    return 0

# Register the function with the database connection, so SQL can call it
con.create_function('fn_CalculateLateFee', 4, fn_CalculateLateFee, deterministic=True)

Step 3: Use the Function in a Query to Calculate Late Fees for Rentals Exceeding the Allowed Duration.

In [ ]:
%%sql
SELECT
    r.rental_id,
    c.customer_id,
    c.first_name || ' ' || c.last_name AS CustomerName,
    r.rental_date,
    r.return_date,
    f.rental_duration,
    fn_CalculateLateFee(r.rental_date, r.return_date, f.rental_duration, 1.50) AS LateFee
FROM
    rental r
    INNER JOIN customer c ON r.customer_id = c.customer_id
    INNER JOIN inventory i ON r.inventory_id = i.inventory_id
    INNER JOIN film f ON i.film_id = f.film_id
WHERE
    fn_CalculateLateFee(r.rental_date, r.return_date, f.rental_duration, 1.50) > 0
ORDER BY
    LateFee DESC;

💻 **Task:** Update the function everywhere it is called so that the late fee per day is now 2.50.

---

## Section 7: 17 Indexes

Step 1: Understanding the Role of Indexes

Adding indexes to database tables is useful when we want to improve the speed and performance of data retrieval operations, particularly in large databases. Indexes act like a database's table of contents, allowing faster searches and queries by providing quick access to rows in tables.

📗 https://learn.microsoft.com/en-us/sql/relational-databases/indexes/indexes

🏢 **Business Scenario:**

Our film rental business needs to quickly access rental and customer data for various reporting and analytical purposes. This assists in enhancing operational efficiency and providing timely insights for decision making.

✅ **Example SQL:** Experiment with creating and using indexes to observe their impact on query performance, especially in large data sets.

Step 1: Drop and recreate Rental and Customer tables for demonstration.

In [ ]:
%%sql
DROP TABLE IF EXISTS FCT_Rental;
DROP TABLE IF EXISTS DIM_Customer;

Step 2: Creating Dimensional Table (DIM_Customer). Create DIM_Customer Table representing a Dimension table in a dimensional data model. Dimension tables store attributes (contextual data) about business entities.

In [ ]:
%%sql
CREATE TABLE DIM_Customer
(
    CustomerID INT PRIMARY KEY,
    CustomerName NVARCHAR(100)
);

Step 3: Populating Dimensional Table. Populate DIM_Customer Table with customer data.

In [ ]:
%%sql
INSERT INTO DIM_Customer (CustomerID, CustomerName)
SELECT customer_id, first_name || ' ' || last_name AS CustomerName
FROM customer;

Step 4: Creating Fact Table (FCT_Rental). Create FCT_Rental Table representing a Fact table in a dimensional data model. Fact tables store quantitative data for analysis and reporting.

In [ ]:
%%sql
CREATE TABLE FCT_Rental
(
    RentalID INT PRIMARY KEY,
    CustomerID INT,
    RentalDate DATETIME,
    ReturnDate DATETIME,
    TotalAmount DECIMAL(5, 2)
);

Step 5: Inflating Fact Table Data. Inflate data in FCT_Rental for a more robust demonstration.

> **SQLite note:** The T-SQL version used a variable and a `WHILE` loop to repeat the `INSERT` 100 times:
>
> ```sql
> DECLARE @i INT = 0;
> WHILE @i < 100 -- Adjust to inflate the table
> BEGIN
>     INSERT INTO FCT_Rental (RentalID, CustomerID, RentalDate, ReturnDate, TotalAmount)
>     SELECT
>         ROW_NUMBER() OVER (ORDER BY r.rental_id) + (SELECT ISNULL(MAX(RentalID), 0) FROM FCT_Rental) AS NewRentalID,
>         r.customer_id, r.rental_date, r.return_date, p.amount
>     FROM rental r
>     JOIN payment p ON r.rental_id = p.rental_id
>     WHERE NOT EXISTS (SELECT 1 FROM FCT_Rental f WHERE f.RentalID = r.rental_id);
>
>     SET @i = @i + 1;
> END;
> ```
>
> SQLite has no variables or loops. Instead, a recursive CTE called `copies` counts from 0 to 99, and the `INSERT` adds one copy of every rental for each number: 100 copies in one statement. Each row gets a new `RentalID` from `ROW_NUMBER()`.
>
> The original loop's `WHERE NOT EXISTS` check also stopped it from inflating the table: after the first pass, almost every `rental_id` was already used as a `RentalID`, so later passes added only a handful of rows. The SQLite version leaves that check out, so the table really does grow 100 times. With about 16,000 rentals, that is about 1.6 million rows, which takes a few seconds to insert.
>
> 📗 https://www.sqlite.org/lang_with.html#recursive_common_table_expressions

In [ ]:
%%sql
WITH RECURSIVE copies(n) AS (
    SELECT 0
    UNION ALL
    SELECT n + 1 FROM copies WHERE n < 99 -- Adjust to inflate the table: 99 makes 100 copies
)
INSERT INTO FCT_Rental (RentalID, CustomerID, RentalDate, ReturnDate, TotalAmount)
SELECT
    ROW_NUMBER() OVER (ORDER BY copies.n, r.rental_id) AS NewRentalID,
    r.customer_id,
    r.rental_date,
    r.return_date,
    p.amount
FROM copies
CROSS JOIN rental r
JOIN payment p ON r.rental_id = p.rental_id;

In [ ]:
%%sql
SELECT COUNT(*) AS rows_in_FCT_Rental
FROM FCT_Rental;

Step 6: Measuring Query Performance Before Indexing. Measure query time before adding index.

> **SQLite note:** T-SQL timed the query with `DECLARE @StartTime DATETIME ... SET @StartTime = GETDATE()` and `DATEDIFF(MILLISECOND, @StartTime, @EndTime)`. Without variables in SQLite, Python's `time.perf_counter()` times the query instead. `%sql` (with one `%`) runs a single line of SQL inside a Python cell.

In [ ]:
import time

start_time = time.perf_counter()
result = %sql SELECT COUNT(*) FROM FCT_Rental JOIN DIM_Customer ON FCT_Rental.CustomerID = DIM_Customer.CustomerID
end_time = time.perf_counter()

# Display execution time
print(result)
print('Execution time before indexing (ms):', round((end_time - start_time) * 1000))

Step 7: Implementing Indexes. Adding index on CustomerID in FCT_Rental to enhance query performance.

In this example, we are using a non-clustered index. Non-clustered indexes are used to improve the performance of queries that search for specific values or ranges of values in a table. They are stored separately from the table data and contain a copy of the indexed columns and a pointer to the corresponding data row. Clustered indexes, on the other hand, physically sort the table data based on the indexed columns. Each table can have only one clustered index, but multiple non-clustered indexes. The choice between clustered and non-clustered indexes depends on factors such as the query patterns, data distribution, and table size.

📗 https://learn.microsoft.com/en-us/sql/relational-databases/indexes/clustered-and-nonclustered-indexes-described

> **SQLite note:** SQLite has no `NONCLUSTERED` keyword: a plain `CREATE INDEX` makes the equivalent of a non-clustered index. (A SQLite table is itself stored in order of its primary key, which is the closest thing to a clustered index.)
>
> 📗 https://www.sqlite.org/lang_createindex.html

In [ ]:
%%sql
CREATE INDEX IDX_CustomerID ON FCT_Rental (CustomerID);

Step 8: Measuring Query Performance After Indexing. Measure query time after adding index.

In [ ]:
start_time = time.perf_counter()
result = %sql SELECT COUNT(*) FROM FCT_Rental JOIN DIM_Customer ON FCT_Rental.CustomerID = DIM_Customer.CustomerID
end_time = time.perf_counter()

# Display execution time
print(result)
print('Execution time after indexing (ms):', round((end_time - start_time) * 1000))

💻 **Task:** Adjust the loop count (the number in `WHERE n < 99` in Step 5) to significantly expand the table size, and observe the impact this has on query performance both before and after adding the index. Run the cells again from Step 1.

---

## SECTION 8: 18 Procedural SQL (Control of Flow Language)

Building on the foundational INSERT INTO statement, control of flow in SQL allows us to make more complex decisions and execute conditional logic based on certain conditions. This approach extends the basic data insertion operation by enabling dynamic and responsive SQL scripting, akin to programming languages. It's particularly useful for complex operations like checking data existence before insertion, handling errors, performing conditional processing, and executing iterative operations within SQL scripts.

📗 https://learn.microsoft.com/en-us/sql/t-sql/language-elements/control-of-flow

🏢 **Business Scenario: Enhancing data insertion with dynamic processing**

Following up on the earlier task of adding new movie categories, we now focus on ensuring data integrity and avoiding duplications in the movie catalogue expansion using control of flow language.

✅ **Example SQL:** Leveraging the control of flow constructs, the SQL script is updated to check for the existence of the 'RomCom' category before inserting it into the 'category' table. This method uses IF...ELSE statements to manage conditional logic effectively, ensuring that the new category is added only if it does not already exist to prevent duplicate entries.

The original T-SQL was:

```sql
-- Check if the 'RomCom' category does not exist in the 'category' table
IF NOT EXISTS (SELECT * FROM dbo.category WHERE name = 'RomCom')
BEGIN
    -- If 'RomCom' does not exist, then insert it into the 'category' table
    INSERT INTO dbo.category (name)
    VALUES ('RomCom');

    -- Declare a variable to hold the number of rows affected by the last operation
    -- @@ROWCOUNT is a system variable that returns the number of rows affected by the last statement
    -- It is commonly used to check if any rows were inserted, updated, or deleted
    DECLARE @RowCount INT = @@ROWCOUNT;

    -- Check if any row was actually added
    IF @RowCount > 0
        -- If a row was added, print the number of rows added
        PRINT CAST(@RowCount AS NVARCHAR) + ' Rows added';
END
ELSE
    -- If 'RomCom' already exists, print that it already exists
    PRINT 'RomCom category already exists';
```

📗 @@ROWCOUNT: https://learn.microsoft.com/en-us/sql/t-sql/functions/rowcount-transact-sql?#examples

> **SQLite note:** SQLite has no control-of-flow language (`IF`, `BEGIN ... END`, `DECLARE`, `PRINT`). The same result comes from two SQLite statements. The `INSERT ... SELECT ... WHERE NOT EXISTS` inserts 'RomCom' only if it is not already there, which is what the `IF NOT EXISTS` did. Then `changes()`, SQLite's equivalent of `@@ROWCOUNT`, gives the number of rows the `INSERT` added, and a `CASE` expression picks the message `PRINT` would have shown.
>
> 📗 https://www.sqlite.org/lang_corefunc.html#changes

In [ ]:
%%sql
-- Insert 'RomCom' into the 'category' table only if it does not already exist
INSERT INTO category (name)
SELECT 'RomCom'
WHERE NOT EXISTS (SELECT * FROM category WHERE name = 'RomCom');

-- changes() returns the number of rows affected by the last statement, like @@ROWCOUNT
SELECT
    CASE
        WHEN changes() > 0 THEN changes() || ' Rows added'
        ELSE 'RomCom category already exists'
    END AS message;

💻 **Task:** Using control of flow language we have made our task "idempotent". This means that the script can be executed multiple times without changing the result beyond the initial application.

Run the code above once, observe the output then run it again.

You will notice that on the first run, it inserts 'RomCom' if it doesn't exist, and on subsequent runs, it simply acknowledges that 'RomCom' already exists without making any changes. This is what idempotent means in the context of SQL scripts: a script that produces the same state in the database, whether run once or multiple times.

*End of SQL examples*